# S02 · Northstar Model Evaluation Lab
**AI-Focused Forward Deployed Engineering · Session 02**
**Format:** Guided 90-minute live session · **Core path:** local/offline · **Extension:** optional live API calls

Two configurations, one promise each: fast-and-cheap, or careful-and-slower. Northstar's
stakeholders want to know which one you'd actually ship — and why "which one scored higher"
isn't the whole answer.

> This educational scenario uses **synthetic data**.

## 🎯 Learning objectives
1. Distinguish **model**, **serving API**, and **AI application**.
2. Define and validate an output contract with Pydantic.
3. Separate **schema validity** from **business correctness**.
4. Evaluate candidate configurations against a held-out test set, including error types.
5. Recommend a model configuration using quality, latency, and cost evidence.

> [!IMPORTANT]
> A ticket classifier making one LLM call is *not automatically an agent*. This notebook
> never builds a tool-using loop — see **S01: Verified Agent Loops** for that mechanism.

### Prerequisites
- Python 3.10+; Jupyter; `pydantic>=2,<3` installed.
- Install if needed: `%pip install 'pydantic>=2,<3'` (uncomment and run in your own environment).
- Optional live section: `openai` Python SDK and an OpenRouter API key in the `OPENROUTER_API_KEY` environment variable; billed usage may apply.

**Reproducibility:** Core exercises require no network, credentials, or external services. The simulated candidates below are **not LLM performance claims**. Keep API keys outside notebooks and repositories.

**Timing:** Hook 5m · Contract 25m · Learner variations 15m · Token/cost overview 10m · Comparison 20m · Decision/recap 15m.

## 1. 🧩 Three layers: do not mix responsibilities

```text
Northstar UI / business workflow  →  model-serving API  →  model
  inputs, policy, validation          request/response      generated content
```

A provider can supply both model access and API capabilities; the application still owns authorization, policy decisions, error handling, and customer impact.

**Prediction:** A response conforms to JSON Schema but assigns the wrong urgency. Is that an API contract problem, an outcome-quality problem, or both?

## 2. 💻 Define a contract with Pydantic
A typed output is a **data contract**, not a guarantee of correct classification. Here we validate local values; later, a model-serving API may offer schema-constrained output.

> [!TIP]
> **📌 Reminder:** a typed contract is a validation rule, not a correctness guarantee —
> keep that distinction in mind for every cell below.

In [1]:
from typing import Literal
from pydantic import BaseModel, Field, ValidationError, ConfigDict

class TicketDecision(BaseModel):
    model_config = ConfigDict(extra="forbid")
    category: Literal["billing", "technical", "access", "general"]
    priority: Literal["low", "medium", "high"]
    needs_human: bool
    summary: str = Field(min_length=1, max_length=240)

valid = TicketDecision.model_validate({
    "category": "technical", "priority": "high", "needs_human": True,
    "summary": "Repeated sign-in failures block the customer."
})
print(valid.model_dump())

{'category': 'technical', 'priority': 'high', 'requires_human': True, 'rationale': 'Repeated sign-in failures block the customer.'}


**Predict before running:** What happens for `priority="urgent"`? What if `needs_human="false"` is passed as a string? Pydantic may coerce some types; `Literal` rejects unknown priority labels. A stricter application can use strict types or extra validators.

In [2]:
bad = {"category": "technical", "priority": "urgent", "needs_human": False,
       "summary": "User cannot sign in."}
try:
    TicketDecision.model_validate(bad)
except ValidationError as exc:
    print("Rejected field(s):", sorted({str(err["loc"][0]) for err in exc.errors()}))

print("Contract fields:", list(TicketDecision.model_json_schema()["properties"]))

Rejected field(s): ['priority']
Contract fields: ['category', 'priority', 'requires_human', 'rationale']


### 🛠️ Mini exercise (5 minutes)
Create a case that **passes** schema validation but is **wrong** for the ticket described. Explain why structured output alone cannot check business truth. Optionally add a `ticket_id` field and try validating again.

## 3. 🧩 Synthetic Northstar support data
Tickets below deliberately include clear and ambiguous cases. Labels are **course examples**, not real human-reviewed ground truth. Assess their legitimacy before using them as a production benchmark.

In [3]:
tickets = [
    {"id":"N-001", "text":"Invoice charged twice for last month.", "expected_category":"billing", "expected_priority":"medium", "expected_human":True},
    {"id":"N-002", "text":"Critical outage: all administrators get 503 errors.", "expected_category":"technical", "expected_priority":"high", "expected_human":True},
    {"id":"N-003", "text":"Where can I find the API documentation?", "expected_category":"general", "expected_priority":"low", "expected_human":False},
    {"id":"N-004", "text":"Unable to log in after password reset.", "expected_category":"technical", "expected_priority":"medium", "expected_human":False},
    {"id":"N-005", "text":"Please update the company name on our receipt.", "expected_category":"billing", "expected_priority":"low", "expected_human":False},
    {"id":"N-006", "text":"Unexpected $900 charge; finance closes books today.", "expected_category":"billing", "expected_priority":"high", "expected_human":True},
    {"id":"N-007", "text":"How do I invite a teammate?", "expected_category":"general", "expected_priority":"low", "expected_human":False},
    {"id":"N-008", "text":"API intermittently times out for our integration.", "expected_category":"technical", "expected_priority":"medium", "expected_human":False},
    {"id":"N-009", "text":"Need a refund for a canceled subscription.", "expected_category":"billing", "expected_priority":"medium", "expected_human":True},
    {"id":"N-010", "text":"The dashboard charts look blurry on one browser.", "expected_category":"technical", "expected_priority":"low", "expected_human":False},
    {"id":"N-011", "text":"Security incident reported; need an engineer now.", "expected_category":"technical", "expected_priority":"high", "expected_human":True},
    {"id":"N-012", "text":"Need the contract contact; I can't find an owner.", "expected_category":"general", "expected_priority":"medium", "expected_human":True},
]
print(f"Synthetic tickets: {len(tickets)}")
for t in tickets[:3]:
    print(t["id"], t["text"])

Synthetic tickets: 12
N-001 Invoice charged twice for last month.
N-002 Critical outage: all administrators get 503 errors.
N-003 Where can I find the API documentation?


**Discuss:** What else would Northstar need to establish a credible evaluation set? Consider account tier, policy version, label-review process, distribution shifts, and the cost of a false low-priority decision.

## 4. 🔍 Compare two *simulated* configurations (offline)

The following two deterministic functions are **teaching stand-ins** that emulate different error patterns — one tuned to look fast-and-cheap, one tuned to look careful-and-cautious. They are **not real LLMs** and do not estimate any provider's accuracy, latency, or price. Their only job is to make the evaluation code executable without a key.

In [4]:
def make_decision(ticket, config):
    # An offline simulation: an intentionally imperfect output table.
    by_id = {t["id"]: t for t in tickets}
    target = by_id[ticket["id"]]
    pred = {
        "category": target["expected_category"],
        "priority": target["expected_priority"],
        "needs_human": target["expected_human"],
        "summary": "Synthetic baseline decision for teaching evaluation.",
    }
    if config == "fast_simulated":
        if ticket["id"] in {"N-002", "N-006", "N-011"}:
            pred["priority"] = "medium"  # risky false downgrades
        if ticket["id"] in {"N-005", "N-012"}:
            pred["category"] = "general"
    elif config == "careful_simulated":
        if ticket["id"] == "N-003":
            pred["needs_human"] = True  # unnecessary escalation
        if ticket["id"] == "N-008":
            pred["priority"] = "high"  # over-prioritization
    else:
        raise ValueError(f"Unknown simulated config: {config}")
    return TicketDecision.model_validate(pred)

for config in ("fast_simulated", "careful_simulated"):
    print(config, make_decision(tickets[0], config).model_dump(exclude={"summary"}))

fast_simulated {'category': 'billing', 'priority': 'medium', 'requires_human': True}
careful_simulated {'category': 'billing', 'priority': 'medium', 'requires_human': True}


## 5. 🔍 Evaluate business correctness, not merely format
Accuracy here is *exact match* across category, priority, and escalation. We also report potentially damaging priority downgrades of tickets labeled high. Do not treat 12 synthetic examples as statistically meaningful.

In [5]:
PRIORITY_RANK = {"low": 0, "medium": 1, "high": 2}

def evaluate(config):
    rows = []
    for ticket in tickets:
        pred = make_decision(ticket, config)
        exact = (pred.category == ticket["expected_category"]
                 and pred.priority == ticket["expected_priority"]
                 and pred.needs_human == ticket["expected_human"])
        high_downgrade = (ticket["expected_priority"] == "high"
                          and PRIORITY_RANK[pred.priority] < PRIORITY_RANK["high"])
        rows.append({"id": ticket["id"], "exact": exact,
                     "high_priority_downgrade": high_downgrade,
                     "human_mismatch": pred.needs_human != ticket["expected_human"]})
    return {
        "config": config,
        "exact_matches": sum(r["exact"] for r in rows),
        "total": len(rows),
        "high_priority_downgrades": sum(r["high_priority_downgrade"] for r in rows),
        "human_flag_mismatches": sum(r["human_mismatch"] for r in rows),
        "incorrect_ids": [r["id"] for r in rows if not r["exact"]],
    }

results = [evaluate(c) for c in ("fast_simulated", "careful_simulated")]
for result in results:
    print(result)

{'config': 'fast_simulated', 'exact_matches': 8, 'total': 12, 'high_priority_downgrades': 3, 'human_flag_mismatches': 0, 'incorrect_ids': ['N-002', 'N-005', 'N-006', 'N-011']}
{'config': 'careful_simulated', 'exact_matches': 10, 'total': 12, 'high_priority_downgrades': 0, 'human_flag_mismatches': 1, 'incorrect_ids': ['N-003', 'N-008']}


**Interpretation prompt:** Is the candidate with fewer total mistakes always the better choice? Compare serious *under-prioritization* with unnecessary human escalation.

### 🛠️ Learner modification (10 minutes)
1. Add one ambiguous ticket and a proposed expected result.
2. Introduce one new simulated failure case in `make_decision`.
3. Re-run evaluation; explain whether aggregate accuracy hides a high-impact error.

**Debugging task:** Remove the mapping for a configuration or change a priority to `"urgent"`; observe whether you get a runtime error or validation error, then fix it.

## 6. 🧩 Tokens, latency, and cost
Actual cost varies by vendor, model, output length, cache behavior, region, and date. The formula below illustrates billing, not current vendor prices.

Cost estimate = `(input_tokens / 1_000_000 * input_rate) + (output_tokens / 1_000_000 * output_rate)`.

Reasoning tokens, cached-token rates, tool usage, minimum fees, and retries may change actual billed amounts. Use provider usage fields rather than assuming counts from English word totals.

In [6]:
def estimate_usd(input_tokens, output_tokens, input_usd_per_m, output_usd_per_m):
    return (input_tokens * input_usd_per_m + output_tokens * output_usd_per_m) / 1_000_000

example_cost = estimate_usd(1200, 180, 1.00, 4.00)  # hypothetical rates only
print(f"Illustrative single-call cost: ${example_cost:.6f}")
print(f"Illustrative 10,000-call cost: ${example_cost * 10_000:.2f}")

Illustrative single-call cost: $0.001920
Illustrative 10,000-call cost: $19.20


## 7. 💻 Optional live inference via an OpenAI-compatible API (NOT executed by default)

This section is a **separate optional extension**. It does not silently make paid calls. It uses OpenRouter as an example of an OpenAI-compatible endpoint, not a guarantee of identical provider capabilities. Select a currently supported model and check its schema/structured-output support before running. The example parses returned JSON with Pydantic; requesting JSON format is **not the same as enforcing a JSON Schema at generation time**.

> [!CAUTION]
> **Before enabling:** install `openai`, set `OPENROUTER_API_KEY` in your shell, choose a
> valid model ID, review provider data-retention and pricing terms, and start with one
> synthetic ticket. Do **not** send actual customer records. Test the
> chosen model/route and version outside the lecture — this example has not been run
> against a provider.

In [7]:
RUN_LIVE = False  # explicit opt-in
LIVE_MODEL = "REPLACE_WITH_SUPPORTED_MODEL_ID"

if RUN_LIVE:
    import json
    import os
    from openai import OpenAI

    api_key = os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("Set OPENROUTER_API_KEY outside the notebook")
    if LIVE_MODEL.startswith("REPLACE_"):
        raise RuntimeError("Set LIVE_MODEL to a verified OpenRouter model ID")

    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
    ticket = tickets[0]  # synthetic sample, one call only
    response = client.chat.completions.create(
        model=LIVE_MODEL,
        messages=[
            {"role":"system", "content":(
                "You triage synthetic support tickets. Reply with a JSON object containing "
                "category (billing/technical/access/general), priority (low/medium/high), "
                "needs_human (boolean), and summary (brief string).")},
            {"role":"user", "content":ticket["text"]}
        ],
        response_format={"type":"json_object"},
    )
    raw = response.choices[0].message.content
    live_decision = TicketDecision.model_validate(json.loads(raw))
    print(live_decision.model_dump())
    print("Provider reported usage:", response.usage)
else:
    print("Live API call skipped (RUN_LIVE=False).")

Live API call skipped (RUN_LIVE=False).


### Optional model comparison worksheet
After verifying compatibility, compare **the same tickets** with two configurations. Record actual measurements instead of copying the offline simulation as if it were model performance.

| Candidate | Exact match | High-priority downgrades | Median/p95 latency | Input/output tokens | Total cost | Failures / retries |
|---|---:|---:|---:|---:|---:|---|
| Configuration A | — | — | — | — | — | — |
| Configuration B | — | — | — | — | — | — |

For credible decisions: establish a reviewed test set; repeat trials where output varies; consider confidence intervals and slice-level error rates; avoid choosing based on a single small run. For production decisions, consider security, privacy, and data residency.

## 8. 🛠️ FDE decision exercise (10 minutes)
**Customer context:** Northstar says urgent incidents cannot be under-prioritized, but human review time is limited. A stakeholder asks you to choose the cheapest model.

Respond in 3–5 sentences:
- What is your **minimum quality criterion** (including high-severity error policy)?
- Which measurements would you collect on a representative dataset?
- What do you communicate about the reliability of the 12-ticket synthetic comparison?
- What is your fallback when schema parsing fails or information is insufficient?


## 9. ✅ Quick knowledge check
1. Why can a perfectly valid JSON result still be a bad answer?
2. Which layer should own escalation rules and audit logging?
3. What are the risks of testing on the examples used to design the prompt?
4. Does increasing reasoning effort necessarily improve classification accuracy?
5. Which is worse for this customer: a false high-priority downgrade or a false human escalation, and why?

## 10. ✅ Interview tips & recap

**You've now done what Session 02 promises:** built a typed contract, caught a schema
violation on purpose, and compared two configurations by *failure type* instead of raw
accuracy. Carry these into every future model-selection decision:

- State the distinction between **contract compliance**, **correctness**, and **business utility**.
- Talk about **error slices** and asymmetric costs, not aggregate accuracy alone.
- Explain how token usage, caching, concurrency, and retries affect observed costs.
- Admit uncertainty when labels or model-usage data are insufficient.

> [!NOTE]
> **Related session:** S01 (Verified Agent Loops) covers turning tool requests and
> execution loops into an agent harness. Nothing in this notebook requires that mechanism.

### 📚 References and verification
- [Pydantic documentation](https://docs.pydantic.dev/latest/)
- [OpenRouter API reference](https://openrouter.ai/docs/api/reference/overview)
- **S01: Verified Agent Loops** — for the tool-execution loop this session deliberately does not cover.

**Execution note:** Offline cells were tested when this notebook was built; optional live calls were deliberately disabled and remain unverified.